In [3]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import BertTokenizer, BertModel,AutoTokenizer, AutoModel

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_CSV = "data.csv"
OUTPUT_CSV = "embeddings.csv"

TEXT_COLUMN = "text"

NUM_TRANSCRIPTS = 1000      # None -> Process entire dataset

RANDOM_SEED = 42

MAX_LENGTH = 512            # BERT maximum
CHUNK_SIZE = 510            # 510 + CLS + SEP = 512

BATCH_SIZE = 64           # Increase if GPU has more VRAM

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(INPUT_CSV)

if NUM_TRANSCRIPTS is not None:
    df = (
        df.sample(
            n=min(NUM_TRANSCRIPTS, len(df)),
            random_state=RANDOM_SEED
        )
        .reset_index(drop=True)
    )

print(f"Transcripts selected : {len(df)}")

# ============================================================
# LOAD MODEL
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

model = BertModel.from_pretrained("bert-base-uncased")

# tokenizer = AutoTokenizer.from_pretrained(
#     "sentence-transformers/all-mpnet-base-v2"
# )

# model = AutoModel.from_pretrained(
#     "sentence-transformers/all-mpnet-base-v2"
# )

model.to(device)
model.eval()

# ============================================================
# BUILD ALL CHUNKS
# ============================================================

all_chunks = []
transcript_map = []

print("\nCreating chunks...")

for idx, text in tqdm(enumerate(df[TEXT_COLUMN]),
                      total=len(df)):

    text = str(text)

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    for i in range(0, len(tokens), CHUNK_SIZE):

        chunk = tokens[i:i + CHUNK_SIZE]

        chunk = (
            [tokenizer.cls_token_id]
            + chunk +
            [tokenizer.sep_token_id]
        )

        all_chunks.append(chunk)

        transcript_map.append(idx)

print(f"\nTotal chunks : {len(all_chunks)}")

# ============================================================
# GENERATE EMBEDDINGS
# ============================================================

chunk_embeddings = []

print("\nGenerating chunk embeddings...")

for start in tqdm(
    range(0, len(all_chunks), BATCH_SIZE)
):

    batch = all_chunks[start:start+BATCH_SIZE]

    encoding = tokenizer.pad(
        {"input_ids": batch},
        padding=True,
        return_tensors="pt"
    )

    encoding = {
        k: v.to(device)
        for k, v in encoding.items()
    }

    with torch.no_grad():

        outputs = model(**encoding)

        # Mean Pooling over tokens
        embeddings = outputs.last_hidden_state.mean(dim=1)

    chunk_embeddings.extend(
        embeddings.cpu().numpy()
    )

# ============================================================
# COMBINE CHUNKS OF SAME TRANSCRIPT
# ============================================================

print("\nCombining chunks...")

transcript_embeddings = [[] for _ in range(len(df))]

for emb, transcript_idx in zip(
    chunk_embeddings,
    transcript_map
):

    transcript_embeddings[transcript_idx].append(emb)

final_embeddings = []

for emb_list in transcript_embeddings:

    emb = np.mean(
        emb_list,
        axis=0
    )

    final_embeddings.append(emb)

# ============================================================
# SAVE
# ============================================================

embedding_df = pd.DataFrame(final_embeddings)

embedding_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\n====================================")
print("Finished Successfully!")
print("====================================")
print("Embeddings Shape :", embedding_df.shape)
print("Saved :", OUTPUT_CSV)

Transcripts selected : 1000
Device : cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Creating chunks...


100%|██████████| 1000/1000 [00:01<00:00, 543.75it/s]



Total chunks : 2536

Generating chunk embeddings...


100%|██████████| 40/40 [01:19<00:00,  1.99s/it]



Combining chunks...

Finished Successfully!
Embeddings Shape : (1000, 768)
Saved : embeddings.csv
